Accompanying Code for: <br>
**Johannes Zeitler, Meinard Müller. "A Unified Perspective on CTC and SDTW using Differentiable DTW", submitted to IEEE Transactions on Audio, Speech, and Language Processing, 2025.**

Johannes Zeitler (johannes.zeitler@audiolabs-erlangen.de), 2025

### Description
Retrieve audio recordings in a database containing a given musical theme. Retrieval strategy based on (Zalkow and Mueller, 2021)

In [1]:
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import librosa
from libfmp import c3
from hcqt import compute_hcqt, compute_hopsize_cqt
from scipy.interpolate import interp1d

from sklearn.metrics import pairwise_distances

from joblib import Parallel, delayed
from itertools import product

from time import time

/home/jzeitler@alabsad.fau.de/anaconda3/envs/dDTW_CTC_fixedVersions/lib/python3.11/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


### get models with best val loss

In [2]:
model_path = "./saved_models"
log_path = "./logs"

configs = []

# horizontal step weight (1-0) = 1.0
config = {"ID": "CTC-A",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [1., 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":1.0}
configs.append(config)

config = {"ID": "CTC-B",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [1., 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":2.0}
configs.append(config)

config = {"ID": "CTC-C",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [1., 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":1e20}
configs.append(config)

config = {"ID": "SDTW-D",
          "loss_variant": "SDTW",
          "step_sizes": [[1,0], [1,1]],
          "step_weights": [1., 1.],
          "gamma": 1.0,
          "min_function": "softmin"}
configs.append(config)

config = {"ID": "SDTW-E",
          "loss_variant": "SDTW",
          "step_sizes": [[1,0], [0,1], [1,1]],
          "step_weights": [1., 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin"}
configs.append(config)

#########################################################
# horizontal step weight (1-0) = 0.1

config = {"ID": "CTC-A-W",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [0.1, 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":1.0}
configs.append(config)

config = {"ID": "CTC-B-W",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [0.1, 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":2.0}
configs.append(config)

config = {"ID": "CTC-C-W",
          "loss_variant": "CTC",
          "step_sizes": [[1,0], [1,1], [1,2]],
          "step_weights": [0.1, 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin",
          "blank_penalty_weight":1e20}
configs.append(config)

config = {"ID": "SDTW-D-W",
          "loss_variant": "SDTW",
          "step_sizes": [[1,0], [1,1]],
          "step_weights": [0.1, 1.],
          "gamma": 1.0,
          "min_function": "softmin"}
configs.append(config)

config = {"ID": "SDTW-E-W",
          "loss_variant": "SDTW",
          "step_sizes": [[1,0], [0,1], [1,1]],
          "step_weights": [0.1, 1., 1.],
          "gamma": 1.0,
          "min_function": "softmin"}
configs.append(config)
###############################################



# EM
config = {"ID": "EM",
          "loss_variant": "EM",
          "step_sizes": [[1,0], [0,1], [1,1]],
          "step_weights": [1.0, 1.0, 1.5],
          "gamma": 0,
          "min_function": "hardmin"}
configs.append(config)

###################################################

# strong
config = {"ID": "strong",
          "loss_variant": "strong",
          "step_sizes": [],
          "step_weights": [],
          "gamma": 0,
          "min_function": ""}
configs.append(config)

In [3]:
n_runs = 5

for config in configs:
    val_losses = []
    for r in range(n_runs):
        
        exp_name = config["ID"] + "_run%i"%(r)

        if not os.path.isfile(os.path.join(model_path, "model_%s_bestVal.pt"%(exp_name))):
            print(exp_name)

        log_in = pd.read_csv(os.path.join("logs", exp_name+"_stats.csv"), sep=";")

        val_losses.append(np.min(log_in.val_loss))

    config["best_val_run"] = np.argmin(val_losses)

### Configure Splits

In [4]:
hop_cqt, fs_cqt = compute_hopsize_cqt(25, 22050, 7)
print(fs_cqt)

24.609375


In [8]:
def get_files_for_split(path, idx):
    files = []
    for i in idx:
        split_in = pd.read_csv(os.path.join(path, "MTD_split5-%i.csv"%(i)))

        for _, row in split_in.iterrows():
            filename = row[0].split(".")[0]
            if os.path.isfile(os.path.join("data", "hcqt", filename+".npy")):
                files.append(filename)
    return files

In [ ]:
hcqt_save_dir = os.path.join("data", "hcqt_database")
MTD_metadata = pd.read_csv(os.path.join("data", "MTD.csv"), sep=";")
split_dir = (os.path.join("data", "MTD_splits"))

splits_test = [5]
files_test = get_files_for_split(split_dir, splits_test)

### Define the model

In [10]:
class ThemeEnhancer(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=6, out_channels=64, kernel_size=(3,3), stride=(1,1), padding="same")
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=(3,3), stride=(1,1), padding="same")
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(3,3), stride=(1,1), padding="same")
        self.conv4 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(3,3), stride=(1,1), padding="same")
        self.conv5 = nn.Conv2d(in_channels=32, out_channels=8, kernel_size=(3,42), stride=(1,1), padding="same")
        self.conv6 = nn.Conv2d(in_channels=8, out_channels=1, kernel_size=(1,1), stride=(1,1), padding="same")

        self.pool1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(1,3), stride=(1,3), padding="valid")
        self.pool1.weight = nn.Parameter(torch.ones(1,1,1,3))
        self.pool1.bias = nn.Parameter(torch.tensor([0.]))

        self.pool2 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(1,61), stride=(1,1), padding="valid")
        pool2_weights = torch.zeros(1,1,1,61)
        for i in range(61):
            if (i%12)==0:
                pool2_weights[0,0,0,i] = 1
        self.pool2.weight = nn.Parameter(pool2_weights)
        self.pool2.bias = nn.Parameter(torch.tensor([0.]))

        self.pool1.weight.requires_grad=False
        self.pool1.bias.requires_grad=False
        self.pool2.weight.requires_grad=False
        self.pool2.bias.requires_grad=False

        self.convEps = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(1,216), stride=(1,1), padding="valid")
    
        

    def forward(self, x):

        column_norm = torch.clamp(torch.sum(x**2, axis=-1, keepdims=True), min=1e-10)
        y = x / column_norm
        
        y = F.leaky_relu(self.conv1(y))
        y = F.leaky_relu(self.conv2(y))
        y = F.leaky_relu(self.conv3(y))
        y = F.leaky_relu(self.conv4(y))
        y = F.leaky_relu(self.conv5(y))
        y = F.sigmoid(self.conv6(y))

        out_chroma = self.pool1(y)
        pred_chroma = self.pool2(out_chroma)
        eps = self.convEps(y)
        pred_CTC = torch.cat([pred_chroma, eps], axis=-1)
        
        return pred_chroma[:,0,:,:], pred_CTC[:,0,:,:], F.softmax(pred_chroma[:,0,:,:], dim=-1), F.softmax(pred_CTC[:,0,:,:], dim=-1)  

### set up dataloaders

In [11]:
fileList_db = [f.split(".")[0] for f in os.listdir(hcqt_save_dir) if ".npy" in f]
fileList_db.sort()

fileList_db = fileList_db
print(len(fileList_db))

10


In [12]:
test_queries_in_db = []

for test_file in files_test:
    MTDID = test_file.split("_")[0][3:]
    metadata = MTD_metadata[MTD_metadata.MTDID==MTDID]
    filename_db = metadata.WAV_WCM.item().split(".")[0]

    if filename_db in fileList_db:
        test_queries_in_db.append(test_file)

test_queries_in_db.sort()

test_queries_in_db = test_queries_in_db
print(len(test_queries_in_db))

2


In [13]:
query_db_match = np.zeros((len(test_queries_in_db), len(fileList_db)))

for i_query, file_query in enumerate(test_queries_in_db):
    MTDID = file_query.split("_")[0][3:]
    metadata = MTD_metadata[MTD_metadata.MTDID==MTDID]

    WCM_query = metadata.WAV_WCM.item().split(".")[0]
    
    for j_db, file_db in enumerate(fileList_db):
        if WCM_query == file_db:
            query_db_match[i_query, j_db] = 1

assert np.prod(np.sum(query_db_match, axis=-1))==1

In [14]:
query_list = []
for file_query in tqdm(test_queries_in_db):
    chroma_query = np.load(os.path.join("data", "weak_targets_stretched_chroma", file_query+".npy")).T
    query_list.append([file_query, chroma_query])

  0%|          | 0/2 [00:00<?, ?it/s]

In [15]:
query_indices = {}
for i, file in enumerate(test_queries_in_db):
    query_indices[file] = i

db_indices = {}
for i, file in enumerate(fileList_db):
    db_indices[file] = i

### matching

In [16]:
def match_chromagrams_with_filename(file_query, chroma_query, file_db, chroma_db):

    C = pairwise_distances(chroma_query, chroma_db, metric="cosine")
    D, wp = librosa.sequence.dtw(C=C,
                             step_sizes_sigma = np.array([[2, 1], [1, 2], [1, 1], ]),
                            weights_add = np.array([0, 0, 0]),
                            weights_mul = np.array([2, 1, 1]),
                             subseq=True,
                             backtrack=True)
    cur_mf = D[-1,:]
    nf_min = np.nanmin(cur_mf)

    result = {"query": file_query,
              "db": file_db,
              "score": nf_min,
              "wp_len": len(wp),
              "query_len": chroma_query.shape[0]}
    

    return result

In [17]:
def model_predict(model, hcqt_in, max_len = 2**13):
    outputs = []
    n_iter = int(np.ceil(hcqt_in.shape[2] / max_len))
    for i in range(n_iter):
        curr_snip = torch.tensor(hcqt_in[:,:,int(i*max_len):int((i+1)*max_len)], dtype=torch.float32, device="cuda:0")    
        output = model(curr_snip)[2]    
        outputs.append(output[0].detach().cpu().numpy())

    return np.concatenate(outputs)

def model_predict_CTC(model, hcqt_in, max_len = 2**13):
    outputs = []
    n_iter = int(np.ceil(hcqt_in.shape[2] / max_len))
    for i in range(n_iter):
        curr_snip = torch.tensor(hcqt_in[:,:,int(i*max_len):int((i+1)*max_len)], dtype=torch.float32, device="cuda:0")    
        output = model(curr_snip)[3][:,:,:-1] # remove blank from ctc probabilities    
        outputs.append(output[0].detach().cpu().numpy())

    return np.concatenate(outputs)

In [ ]:
# compute top-k rank
def rank(x, axis=None):
    return np.argsort(np.argsort(x, axis=axis), axis=axis)

In [ ]:
for config in configs:
    print("Evaluating for " + config["ID"])

    # load a model
    model = ThemeEnhancer().to("cuda")
    model.load_state_dict(torch.load(os.path.join("saved_models", "model_%s_run%i_bestVal.pt"%(config["ID"],config["best_val_run"])), weights_only=True))
    model.eval()

    # compute enhanced chromagram for all pieces in the database
    chroma_db_list = []
    for j_db, file_db in tqdm(enumerate(fileList_db)):
        hcqt_in = np.load(os.path.join(hcqt_save_dir, file_db+".npy")).transpose((2,1,0))[None,:]
        
        if "CTC" in config["loss_variant"]:
            chroma_db = model_predict_CTC(model, hcqt_in)
        else:
            chroma_db = model_predict(model, hcqt_in)
    
        chroma_db_list.append([file_db, chroma_db])

    # compute subsequence DTW distance between queries and enhanced database chromagrams
    query_db_pairs = list(product(query_list, chroma_db_list))
    t = time()
    all_results = Parallel(n_jobs=64, verbose=False, prefer="processes")(
            delayed(match_chromagrams_with_filename)(file_query, chroma_query, file_db, chroma_db)
            for (file_query, chroma_query), (file_db, chroma_db)
            in tqdm(query_db_pairs))
    print("elapsed time %.1fs"%(time()-t))

    # normalize subsequende DTW distance by warping path length    
    normalized_scores = np.zeros_like(query_db_match)
    for result in all_results:
        normalized_scores[query_indices[result["query"]], db_indices[result["db"]]] = result["score"] / result["wp_len"]
    
    # compute top-k ratings    
    ratings = (rank(normalized_scores, axis=1) * query_db_match).sum(axis=-1)
    print("mean: %.4f, median: %i"%(np.mean(ratings), np.median(ratings)))
    top_k = [1, 5, 10, 20, 50]
    for k in top_k:
        print("Top %i: %.2f percent"%(k, np.mean(ratings < k)*100))

    # save results
    np.savez("./results/matchings/"+ config["ID"] + ".npz", 
             test_queries_in_db = test_queries_in_db,
             fileList_db = fileList_db,
             all_results = all_results,
             ratings = ratings,
             normalized_scores = normalized_scores)

    print("====================================================")